<a href="https://colab.research.google.com/github/Learn-AIMLOps/Instruction_fine_tunning/blob/main/Generated_IFT_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install pymupdf
!pip install transformers accelerate bitsandbytes peft datasets
!pip install pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [2]:
import torch
import re
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# ===============================
# MODEL A (Domain Model = Base + QLoRA Adapter)
# ===============================
base_model_A = "Qwen/Qwen2-1.5B"
adapter_A = "RohitGu/Qwen2-1.5B-LLMOps-LoRA"  # HF repo name or local path

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer_A = AutoTokenizer.from_pretrained(base_model_A, trust_remote_code=True)
tokenizer_A.pad_token = tokenizer_A.eos_token

model_A = AutoModelForCausalLM.from_pretrained(
    base_model_A,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model_A = PeftModel.from_pretrained(model_A, adapter_A)
model_A.eval()

# ===============================
# MODEL B (Teacher Instruct)
# ===============================
base_model_B = "Qwen/Qwen2-1.5B-Instruct"

tokenizer_B = AutoTokenizer.from_pretrained(base_model_B, trust_remote_code=True)
tokenizer_B.pad_token = tokenizer_B.eos_token

model_B = AutoModelForCausalLM.from_pretrained(
    base_model_B,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model_B.eval()

print("✅ Models loaded successfully")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Models loaded successfully


In [ ]:
def format_chat(tokenizer, user_prompt):
    messages = [
        {"role": "system", "content": "You are a precise and strict AI."},
        {"role": "user", "content": user_prompt}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
def run_model(model, tokenizer, prompt, max_new_tokens=256, temperature=0.4):

    formatted_prompt = format_chat(tokenizer, prompt)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

In [ ]:
def extract_json(text):
    match = re.search(r"<json>(.*?)</json>", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1).strip())
    except:
        return None

In [ ]:
DOMAIN_EXTRACTION_PROMPT = """
You are a strict information extraction system.

Extract ONLY factual, domain-specific knowledge from the text.

STRICT RULES:
- Output MUST be bullet points.
- Each bullet MUST be a standalone atomic fact.
- No rewriting or summarizing.
- No interpretation.
- No opinions.
- No instructions.
- No assumptions.
- No meta text (chapter names, copyright, references).
- No hallucinations.
- If no domain facts exist, output exactly:
NO_FACTS

TEXT:
{chunk}

FACTS:
-
"""

In [ ]:
IFT_GENERATION_PROMPT = """
You are generating high-quality instruction fine-tuning data.

Generate EXACTLY ONE instruction-input-response sample.

STRICT RULES:
- The instruction must require reasoning.
- The instruction must NOT be trivial.
- The response must be fully grounded in the provided facts.
- Do NOT invent information.
- Use ONLY the provided facts.
- Output STRICT JSON.
- No markdown.
- No commentary.
- No explanation.
- No extra text outside JSON.

FACTS:
{facts}

Output ONLY this format:

<json>
{{
  "instruction": "...",
  "input": "...",
  "response": "..."
}}
</json>
"""

In [ ]:
SELF_EVAL_PROMPT = """
You are a strict dataset validator.

Evaluate the following sample.

Reject if:
- Any hallucinated information exists.
- The response uses knowledge not in the facts.
- JSON structure is invalid.
- The instruction is trivial.
- The instruction does not require reasoning.
- The response is incomplete.

Sample:
{sample}

Output ONLY one word:

accept
or
reject
"""

In [3]:
# STEP 2 — Extract text from PDF
# Purpose: Use PyMuPDF to extract clean text from each page.

import fitz

def extract_text(path):
    doc = fitz.open(path)
    pages = []

    for page_number, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page": page_number + 1,
            "text": text.strip()
        })

    return pages

pdf_path = "LLMOps Guide.pdf"
text_pages = extract_text(pdf_path)

len(text_pages), text_pages[0]

(190,
 {'page': 1,
  'text': '1 S T  E D I T I O N\nImplementing eff ective strategies for Large Language Models \nin deployment and continuous improvement\nEssential Guide to LLMOps\nRYAN DOAN'})

In [4]:
# STEP 3 — Robust image extraction for ANY PDF (Packt, O'Reilly, ACM, etc.)
# Purpose: Extract embedded images safely by converting all colorspaces to RGB.

import fitz
import os

def extract_images(path, out_dir="pdf_images"):
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(path)
    images_info = []

    for page_number, page in enumerate(doc):
        image_list = page.get_images(full=True)

        for img_index, img in enumerate(image_list):
            xref = img[0]

            try:
                pix = fitz.Pixmap(doc, xref)

                # Case 1: CMYK or other unsupported colorspaces
                if pix.colorspace is None:
                    raise ValueError("Image has no colorspace (likely a mask).")

                if pix.colorspace.n not in (1, 3):
                    # Convert ANY non-RGB colorspace to RGB
                    pix = fitz.Pixmap(fitz.csRGB, pix)

                # Case 2: Alpha channel present → remove alpha
                if pix.alpha:
                    pix = fitz.Pixmap(fitz.csRGB, pix)

                img_path = f"{out_dir}/page{page_number+1}_img{img_index}.png"
                pix.save(img_path)

                images_info.append({
                    "page": page_number + 1,
                    "image_path": img_path
                })

                pix = None  # free memory

            except Exception as e:
                print(f"Skipping image on page {page_number+1}, index {img_index}: {e}")
                continue

    return images_info

images_info = extract_images(pdf_path)
len(images_info), images_info[:5]


(41,
 [{'page': 1, 'image_path': 'pdf_images/page1_img0.png'},
  {'page': 1, 'image_path': 'pdf_images/page1_img1.png'},
  {'page': 1, 'image_path': 'pdf_images/page1_img2.png'},
  {'page': 1, 'image_path': 'pdf_images/page1_img3.png'},
  {'page': 1, 'image_path': 'pdf_images/page1_img4.png'}])

In [5]:
# STEP 4 — Smart captioning for extracted images using Qwen2-VL-2B-Instruct
# Purpose: Caption only meaningful diagrams using the correct Qwen2-VL model class.

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from PIL import Image
import torch

vl_model_name = "Qwen/Qwen2-VL-2B-Instruct"

# Load processor and model
processor = AutoProcessor.from_pretrained(vl_model_name)

vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
    vl_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

def is_meaningful_image(image_path, min_size=200):
    try:
        img = Image.open(image_path)
        w, h = img.size

        if w < min_size or h < min_size:
            return False

        if img.mode in ["1", "L"]:  # grayscale or binary → usually not diagrams
            return False

        if img.getbbox() is None:   # blank/transparent
            return False

        return True
    except Exception:
        return False


def caption_image(image_path):
    if not is_meaningful_image(image_path):
        return None

    try:
        image = Image.open(image_path).convert("RGB")
        prompt = "Describe this diagram in detail."

        # Prepare inputs
        inputs = processor(text=prompt, images=image, return_tensors="pt")
        inputs = {k: v.to(vl_model.device) for k, v in inputs.items()}

        # -------------------------------
        # FIX 1: Removed undefined variable `do_sample`
        # Old: temperature=0.0 if not do_sample else 0.7
        # New: deterministic captioning (do_sample=False, temperature=0.0)
        # -------------------------------
        output = vl_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,     # <-- FIXED: always deterministic
            temperature=0.0      # <-- FIXED: removed dependency on undefined do_sample
        )
        # -------------------------------

        # Trim prompt tokens
        generated_ids = output[0][inputs["input_ids"].shape[1]:]
        caption = processor.decode(generated_ids, skip_special_tokens=True).strip()

        return caption

    except Exception as e:
        print(f"Skipping caption for {image_path}: {e}")
        return None


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [6]:
# STEP 5 — Merge text + image captions into unified JSON
# Purpose: Create a structured domain dataset for continued pretraining.

def merge_text_and_images(text_pages, images_info):
    merged = []

    for page in text_pages:
        page_num = page["page"]
        page_images = [img for img in images_info if img["page"] == page_num]

        merged.append({
            "page": page_num,
            "text": page["text"],
            "images": [
                {
                    "path": img["image_path"],
                    "caption": caption_image(img["image_path"])
                }
                for img in page_images
                if caption_image(img["image_path"]) is not None
            ]
        })

    return merged
merged_pages = merge_text_and_images(text_pages, images_info)
merged_pages[0]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Skipping caption for pdf_images/page1_img0.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_img1.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_img2.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_img3.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_img4.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_img5.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_img6.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_img7.png: Image features and image tokens do not match, tokens: 0, features: 1156
Skipping caption for pdf_images/page1_im

{'page': 1,
 'text': '1 S T  E D I T I O N\nImplementing eff ective strategies for Large Language Models \nin deployment and continuous improvement\nEssential Guide to LLMOps\nRYAN DOAN',
 'images': []}

In [7]:
# STEP 5 — Merge text + image captions into unified JSON
# Purpose: Create a structured domain dataset for continued pretraining.
# FIXES:
# 1. Avoid calling caption_image() twice (major slowdown)
# 2. Add logging for captioned vs skipped images
# 3. Store caption result in a variable before filtering

def merge_text_and_images(text_pages, images_info):
    merged = []

    for page in text_pages:
        page_num = page["page"]
        page_images = [img for img in images_info if img["page"] == page_num]

        print(f"\n--- Processing Page {page_num} ---")  # LOGGING

        image_entries = []
        for img in page_images:

            img_path = img["image_path"]

            # -------------------------------
            # FIX 1: Call caption_image() ONCE
            # -------------------------------
            caption = caption_image(img_path)

            if caption is None:
                print(f"SKIPPED: {img_path}")  # LOGGING
                continue

            print(f"CAPTIONED: {img_path}")  # LOGGING

            image_entries.append({
                "path": img_path,
                "caption": caption
            })

        merged.append({
            "page": page_num,
            "text": page["text"],
            "images": image_entries
        })

    return merged


# Run Step 5
merged_pages = merge_text_and_images(text_pages, images_info)

# Show first page result
merged_pages[0]



--- Processing Page 1 ---
Skipping caption for pdf_images/page1_img0.png: Image features and image tokens do not match, tokens: 0, features: 1156
SKIPPED: pdf_images/page1_img0.png
Skipping caption for pdf_images/page1_img1.png: Image features and image tokens do not match, tokens: 0, features: 1156
SKIPPED: pdf_images/page1_img1.png
Skipping caption for pdf_images/page1_img2.png: Image features and image tokens do not match, tokens: 0, features: 1156
SKIPPED: pdf_images/page1_img2.png
Skipping caption for pdf_images/page1_img3.png: Image features and image tokens do not match, tokens: 0, features: 1156
SKIPPED: pdf_images/page1_img3.png
Skipping caption for pdf_images/page1_img4.png: Image features and image tokens do not match, tokens: 0, features: 1156
SKIPPED: pdf_images/page1_img4.png
Skipping caption for pdf_images/page1_img5.png: Image features and image tokens do not match, tokens: 0, features: 1156
SKIPPED: pdf_images/page1_img5.png
Skipping caption for pdf_images/page1_img6.

{'page': 1,
 'text': '1 S T  E D I T I O N\nImplementing eff ective strategies for Large Language Models \nin deployment and continuous improvement\nEssential Guide to LLMOps\nRYAN DOAN',
 'images': []}

In [8]:
# Step 6 — Chunk Domain Data for Continued Pretraining

# STEP 6 — Chunk domain data
# Purpose: Convert merged pages into training chunks for continued pretraining.

import json

def create_chunks(merged_pages, max_chars=3000):
    chunks = []
    buffer = ""

    for page in merged_pages:
        page_text = page["text"]
        captions = "\n".join([img["caption"] for img in page["images"]])

        combined = page_text + "\n" + captions

        if len(buffer) + len(combined) < max_chars:
            buffer += combined + "\n"
        else:
            chunks.append(buffer.strip())
            buffer = combined + "\n"

    if buffer.strip():
        chunks.append(buffer.strip())

    return chunks

chunks = create_chunks(merged_pages)
len(chunks), chunks[0][:500]


(153,
 '1 S T  E D I T I O N\nImplementing eff ective strategies for Large Language Models \nin deployment and continuous improvement\nEssential Guide to LLMOps\nRYAN DOAN\n\nEssential Guide to LLMOps\nImplementing eﬀ ective strategies for Large Language Models \nin deployment and continuous improvement\nRyan Doan\n\nEssential Guide to LLMOps\nCopyright © 2024 Packt Publishing\nAll rights reserved. No part of this book may be reproduced, stored in a retrieval system, or transmitted \nin any form or by any means, with')

In [9]:
import json

def create_chunks(merged_pages, max_chars=3000):
    chunks = []
    buffer = ""

    for page in merged_pages:
        page_text = page["text"]
        captions = "\n".join([img["caption"] for img in page["images"]])
        combined = page_text + "\n" + captions

        if len(buffer) + len(combined) < max_chars:
            buffer += combined + "\n"
        else:
            if buffer.strip():           # avoid saving empty chunks
                chunks.append(buffer.strip())
            buffer = combined + "\n"

    if buffer.strip():
        chunks.append(buffer.strip())

    return chunks


# ────────────────────────────────────────────────
# Create chunks
chunks = create_chunks(merged_pages, max_chars=3000)

print(f"Created {len(chunks)} chunks")
print(f"First chunk preview:\n{chunks[0][:500]}...\n")

# Option A: Simple list of strings (most common for pretraining)
with open("training_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=1)
    #               ↑↑↑ this is usually what you want

print("Saved to: training_chunks.json")

Created 153 chunks
First chunk preview:
1 S T  E D I T I O N
Implementing eff ective strategies for Large Language Models 
in deployment and continuous improvement
Essential Guide to LLMOps
RYAN DOAN

Essential Guide to LLMOps
Implementing eﬀ ective strategies for Large Language Models 
in deployment and continuous improvement
Ryan Doan

Essential Guide to LLMOps
Copyright © 2024 Packt Publishing
All rights reserved. No part of this book may be reproduced, stored in a retrieval system, or transmitted 
in any form or by any means, with...

Saved to: training_chunks.json


In [10]:
print(type(chunks))

<class 'list'>


In [11]:
print("Total chunks:", len(chunks))

Total chunks: 153


In [ ]:
chunks[40]

'Continuous improvement\n37\nincorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of \nthe LLM. $ese monitoring processes not only aid in maintaining the quality of the model but also \nserve as a foundation for continuous improvement, feeding into the model’s ongoing training and \ndevelopment to enhance its performance over time.\nContinuous improvement\nIn the realm of LLMs such as GPT, continuous improvement is paramount to maintaining their \nrelevance and e%cacy. $is process is multifaceted and involves integrating new data, developing \nstrategies to prevent previous knowledge from being forgotten, and incorporating human feedback.\nFirstly, incorporating newer data is essential as language and societal contexts evolve. Regular updates \nwith recent datasets ensure that models grasp current terminologies, phrases, and relevant topics. \n$is could involve data from diverse sources, such as the latest news articles, scienti!c public

In [ ]:
print("========== FIRST SAMPLE VALIDATION ==========")

first_chunk = chunks[10]   # You can change index if needed

# 1️⃣ Extract facts
facts = run_model(model_A, tokenizer_A, DOMAIN_EXTRACTION_PROMPT.format(chunk=first_chunk))

print("\n--- Extracted Facts ---\n")
print(facts)

if "NO_FACTS" in facts or len(facts.strip()) < 30:
    raise ValueError("❌ No usable domain facts.")

# 2️⃣ Generate IFT
raw_sample = run_model(
    model_B,
    tokenizer_B,
    IFT_GENERATION_PROMPT.format(facts=facts),
    max_new_tokens=300
)

print("\n--- Raw Teacher Output ---\n")
print(raw_sample)

sample_json = extract_json(raw_sample)

print("\n--- Parsed JSON ---\n")
print(sample_json)

if sample_json is None:
    raise ValueError("❌ JSON malformed.")

# 3️⃣ Self Eval
verdict = run_model(
    model_B,
    tokenizer_B,
    SELF_EVAL_PROMPT.format(sample=json.dumps(sample_json)),
    max_new_tokens=10,
    temperature=0.0
)

print("\n--- Self Evaluation ---\n")
print(verdict)

if "accept" not in verdict.lower():
    raise ValueError("❌ First sample rejected.")

print("\n✅ FIRST SAMPLE PASSED")

========== FIRST SAMPLE VALIDATION ==========



--- Extracted Facts ---

system
You are a precise and strict AI.
user

You are a strict information extraction system.

Extract ONLY factual, domain-specific knowledge from the text.

STRICT RULES:
- Output MUST be bullet points.
- Each bullet MUST be a standalone atomic fact.
- No rewriting or summarizing.
- No interpretation.
- No opinions.
- No instructions.
- No assumptions.
- No meta text (chapter names, copyright, references).
- No hallucinations.
- If no domain facts exist, output exactly:
NO_FACTS

TEXT:
Introduction to LLMs and LLMOps
6
$e impact of RNNs and LSTMs was particularly profound in machine translation. $e introduction 
of sequence-to-sequence (Seq2Seq) learning, which employed an encoder-decoder framework, 
revolutionized this !eld. Google’s Neural Machine Translation system exempli!ed this, translating 
entire sentences with contextual integrity, surpassing traditional phrase-based systems.
LSTMs also excelled in text generation, producing coherent, contextually r

ValueError: ❌ No usable domain facts.

In [ ]:
print("=== Chunk 40 preview ===")
print(chunks[40][:1200])          # first ~1200 chars
print("\n…\n")
print(chunks[40][-400:])          # last part
print(f"\nLength: {len(chunks[40])} chars")

=== Chunk 40 preview ===
Continuous improvement
37
incorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of 
the LLM. $ese monitoring processes not only aid in maintaining the quality of the model but also 
serve as a foundation for continuous improvement, feeding into the model’s ongoing training and 
development to enhance its performance over time.
Continuous improvement
In the realm of LLMs such as GPT, continuous improvement is paramount to maintaining their 
relevance and e%cacy. $is process is multifaceted and involves integrating new data, developing 
strategies to prevent previous knowledge from being forgotten, and incorporating human feedback.
Firstly, incorporating newer data is essential as language and societal contexts evolve. Regular updates 
with recent datasets ensure that models grasp current terminologies, phrases, and relevant topics. 
$is could involve data from diverse sources, such as the latest news articles, scie

In [ ]:
facts = run_model(model_A, tokenizer_A, DOMAIN_EXTRACTION_PROMPT.format(chunk=first_chunk))
print("\n--- Extracted Facts (raw) ---\n")
print(repr(facts))               # ← use repr() to see whitespace & escapes
print(f"Length: {len(facts.strip())}")


--- Extracted Facts (raw) ---

"system\nYou are a precise and strict AI.\nuser\n\nYou are a strict information extraction system.\n\nExtract ONLY factual, domain-specific knowledge from the text.\n\nSTRICT RULES:\n- Output MUST be bullet points.\n- Each bullet MUST be a standalone atomic fact.\n- No rewriting or summarizing.\n- No interpretation.\n- No opinions.\n- No instructions.\n- No assumptions.\n- No meta text (chapter names, copyright, references).\n- No hallucinations.\n- If no domain facts exist, output exactly:\nNO_FACTS\n\nTEXT:\nIntroduction to LLMs and LLMOps\n6\n$e impact of RNNs and LSTMs was particularly profound in machine translation. $e introduction \nof sequence-to-sequence (Seq2Seq) learning, which employed an encoder-decoder framework, \nrevolutionized this !eld. Google’s Neural Machine Translation system exempli!ed this, translating \nentire sentences with contextual integrity, surpassing traditional phrase-based systems.\nLSTMs also excelled in text generation,

In [ ]:
# ────────────────────────────────────────────────
# Helper: rich pretty-print (much better than plain print in notebooks)
# Install once: !pip install rich   (run in a cell if not installed)
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich import print as rprint   # ← shadow built-in print

console = Console()

def pp(title: str, content, style="bold cyan"):
    """Pretty panel with title"""
    console.print(Panel.fit(
        str(content),
        title=f" [bold]{title}[/bold] ",
        border_style=style,
        padding=(1, 2),
        expand=False
    ))

def pp_md(title: str, md_content):
    console.print(Panel(
        Markdown(md_content),
        title=title,
        border_style="blue",
        expand=True
    ))

In [ ]:
!pip install --upgrade rich

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from rich import print as rprint

console = Console()

In [ ]:
# Pick chunk interactively — change number and re-run cell
CHUNK_IDX = 40          # ← easy to change

first_chunk = chunks[CHUNK_IDX]

# ── Quick inspection ───────────────────────────────────────
pp(f"Chunk #{CHUNK_IDX}  ({len(first_chunk):,} chars)",
   first_chunk[:1200] + "\n…\n" + first_chunk[-300:],
   style="bold green")

# Optional: show word count, line count
lines = first_chunk.count("\n") + 1
words = len(first_chunk.split())
rprint(f"[dim]Lines:[/] {lines}    [dim]Words:[/] {words}    [dim]Chars:[/] {len(first_chunk)}")

# ── Decide whether to skip ────────────────────────────────
MIN_CHARS = 400
MIN_WORDS = 60
MIN_LINES = 5

if len(first_chunk) < MIN_CHARS or words < MIN_WORDS or lines < MIN_LINES:
    pp("Low quality chunk — skipping",
       "Too short / too few lines / too few words",
       style="bold red")
else:
    rprint("[bold green]→ Proceeding with extraction[/bold green]")

TypeError: Panel.fit() got an unexpected keyword argument 'expand'

In [ ]:
# ────────────────────────────────────────────────────────────────
#   Rich pretty-print helpers — safe version (no 'expand' issue)
# ────────────────────────────────────────────────────────────────

from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

def pp(title: str, content, style="bold cyan"):
    """Simple panel — works on old & new rich versions"""
    console.print(Panel(
        str(content),
        title=f" [bold]{title}[/bold] ",
        border_style=style,
        padding=(1, 2),
        # NO expand=... here — we use regular Panel instead of .fit()
    ))

def pp_md(title: str, content: str):
    """Markdown version"""
    console.print(Panel(
        Markdown(content),
        title=f" [bold]{title}[/bold] ",
        border_style="blue dim",
        padding=(1, 2),
    ))

# ────────────────────────────────────────────────────────────────
#   Chunk viewer cell — change CHUNK_IDX and re-run
# ────────────────────────────────────────────────────────────────

CHUNK_IDX = 40          # ← change this number and re-run the cell

try:
    first_chunk = chunks[CHUNK_IDX]
except NameError:
    console.print("[bold red]Error:[/] 'chunks' list not found in namespace")
    raise
except IndexError:
    console.print(f"[bold red]Error:[/] Chunk index {CHUNK_IDX} out of range "
                  f"(valid: 0–{len(chunks)-1})")
    raise

# ── Show the chunk ───────────────────────────────────────────────
pp(
    title=f"Chunk #{CHUNK_IDX}  ({len(first_chunk):,} characters)",
    content=first_chunk[:1400] + "\n…\n" + first_chunk[-400:],
    style="bold green"
)

# Basic stats
lines = first_chunk.count("\n") + 1
words = len(first_chunk.split())
console.print(
    f"[dim]Lines:[/] {lines:4d}    "
    f"[dim]Words:[/] {words:4d}    "
    f"[dim]Characters:[/] {len(first_chunk):,} "
)

# ── Very simple quality gate (you can adjust thresholds) ─────────
MIN_CHARS = 350
MIN_WORDS = 60
MIN_LINES = 4

if len(first_chunk) < MIN_CHARS or words < MIN_WORDS or lines < MIN_LINES:
    console.print(
        "[yellow bold]⚠️  Low quality chunk — probably not worth processing[/yellow bold]"
    )
else:
    console.print("[green bold]→ Looks reasonable — can proceed to extraction[/green bold]")

# Optional: first & last sentence preview
if '.' in first_chunk:
    sentences = first_chunk.split('.')
    console.print("\n[dim]First sentence:[/dim]")
    console.print(sentences[0].strip()[:160] + " …" if len(sentences[0]) > 160 else sentences[0].strip())
    if len(sentences) > 1:
        console.print("\n[dim]Last sentence:[/dim]")
        console.print(sentences[-1].strip()[:160] + " …" if len(sentences[-1]) > 160 else sentences[-1].strip())

╭────────────────────────────────────────  Chunk #40  (2,914 characters)  ────────────────────────────────────────╮
│                                                                                                                 │
│  Continuous improvement                                                                                         │
│  37                                                                                                             │
│  incorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of       │
│  the LLM. $ese monitoring processes not only aid in maintaining the quality of the model but also               │
│  serve as a foundation for continuous improvement, feeding into the model’s ongoing training and                │
│  development to enhance its performance over time.                                                              │
│  Continuous improvement                                                                                         │
│  In the realm of LLMs such as GPT, continuous improvement is paramount to maintaining their                     │
│  relevance and e%cacy. $is process is multifaceted and involves integrating new data, developing                │
│  strategies to prevent previous knowledge from being forgotten, and incorporating human feedback.               │
│  Firstly, incorporating newer data is essential as language and societal contexts evolve. Regular updates       │
│  with recent datasets ensure that models grasp current terminologies, phrases, and relevant topics.             │
│  $is could involve data from diverse sources, such as the latest news articles, scienti!c publications,         │
│  and trending internet content. Such updates help the model stay current with linguistic trends and             │
│  societal changes.                                                                                              │
│  Secondly, a signi!cant challenge in training neural networks, especially when they learn new                   │
│  information, is avoiding catastrophic forgetting. $is phenomenon occurs when a model loses the                 │
│  information it had learned previously upon acquiring new knowledge. Techniques such as elastic                 │
│                                                                                                                 │
│  …                                                                                                              │
│  gies ensure they remain eﬀective                                                                               │
│  tools capable of understanding and generating human language in a manner that is both useful                   │
│  and responsible.                                                                                               │
│  Summary                                                                                                        │
│  $is chapter provided a review of each component in the LLMOps landscape by iterating through                   │
│  data preparation, model development, governance and review, model serving, and monitoring. In the              │
│  next chapter, we’ll dive deeper into data preparation.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Lines:   36    Words:  414    Characters: 2,914

→ Looks reasonable — can proceed to extraction

First sentence:

Continuous improvement
37
incorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of 
the LLM

Last sentence:

In [ ]:
# ────────────────────────────────────────────────────────────────
#   FACT EXTRACTION — with full prompt visibility (debug mode)
# ────────────────────────────────────────────────────────────────

CHUNK_IDX = 40
first_chunk = chunks[CHUNK_IDX]

# ── Show chunk again for reference ─────────────────────────────
pp(
    title=f"Chunk #{CHUNK_IDX}  ({len(first_chunk):,} chars)",
    content=first_chunk[:1400] + "\n…\n" + first_chunk[-400:],
    style="bold green"
)

# ── Build & show the EXACT prompt being sent ───────────────────
try:
    facts_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=first_chunk)
except KeyError as e:
    console.print(f"[bold red]Prompt formatting failed:[/] Missing placeholder {e}")
    raise

# Print first and last part of the prompt (very important!)
console.print("\n[bold magenta]FULL PROMPT SENT TO MODEL_A (first 800 chars):[/bold magenta]")
console.print(facts_prompt[:800] + " …")

console.print("\n[bold magenta]… last 800 chars of prompt:[/bold magenta]")
console.print(" … " + facts_prompt[-800:])

# Optional: rough token estimate (if you have the tokenizer)
try:
    token_len = len(tokenizer_A.encode(facts_prompt))
    console.print(f"\n[bold yellow]Approx. tokens:[/] {token_len}")
    if token_len > 3800:  # adjust to your model's real context length
        console.print("[red bold]!!! PROMPT TOO LONG — chunk probably truncated !!![/red bold]")
except NameError:
    console.print("[dim](tokenizer_A not available — skipping token count)[/dim]")

# ── Run extraction only if prompt looks sane ───────────────────
if "Continuous improvement" not in facts_prompt[-1500:]:
    console.print(
        "[bold red]!!! CRITICAL:[/] The chunk text is NOT present in the prompt → variable bug!"
    )
    console.print("[yellow]Do NOT proceed — fix the .format() or variable first.[/yellow]")
else:
    console.print("\n[green]Chunk text appears in prompt → looks ok[/green]")
    # Uncomment only when ready:
    # facts = run_model(model_A, tokenizer_A, facts_prompt)
    # pp("Extracted Facts (raw)", facts)

╭──────────────────────────────────────────  Chunk #40  (2,914 chars)  ───────────────────────────────────────────╮
│                                                                                                                 │
│  Continuous improvement                                                                                         │
│  37                                                                                                             │
│  incorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of       │
│  the LLM. $ese monitoring processes not only aid in maintaining the quality of the model but also               │
│  serve as a foundation for continuous improvement, feeding into the model’s ongoing training and                │
│  development to enhance its performance over time.                                                              │
│  Continuous improvement                                                                                         │
│  In the realm of LLMs such as GPT, continuous improvement is paramount to maintaining their                     │
│  relevance and e%cacy. $is process is multifaceted and involves integrating new data, developing                │
│  strategies to prevent previous knowledge from being forgotten, and incorporating human feedback.               │
│  Firstly, incorporating newer data is essential as language and societal contexts evolve. Regular updates       │
│  with recent datasets ensure that models grasp current terminologies, phrases, and relevant topics.             │
│  $is could involve data from diverse sources, such as the latest news articles, scienti!c publications,         │
│  and trending internet content. Such updates help the model stay current with linguistic trends and             │
│  societal changes.                                                                                              │
│  Secondly, a signi!cant challenge in training neural networks, especially when they learn new                   │
│  information, is avoiding catastrophic forgetting. $is phenomenon occurs when a model loses the                 │
│  information it had learned previously upon acquiring new knowledge. Techniques such as elastic                 │
│                                                                                                                 │
│  …                                                                                                              │
│  gies ensure they remain eﬀective                                                                               │
│  tools capable of understanding and generating human language in a manner that is both useful                   │
│  and responsible.                                                                                               │
│  Summary                                                                                                        │
│  $is chapter provided a review of each component in the LLMOps landscape by iterating through                   │
│  data preparation, model development, governance and review, model serving, and monitoring. In the              │
│  next chapter, we’ll dive deeper into data preparation.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FULL PROMPT SENT TO MODEL_A (first 800 chars):

You are a strict information extraction system.

Extract ONLY factual, domain-specific knowledge from the text.

STRICT RULES:
- Output MUST be bullet points.
- Each bullet MUST be a standalone atomic fact.
- No rewriting or summarizing.
- No interpretation.
- No opinions.
- No instructions.
- No assumptions.
- No meta text (chapter names, copyright, references).
- No hallucinations.
- If no domain facts exist, output exactly:
NO_FACTS

TEXT:
Continuous improvement
37
incorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of 
the LLM. $ese monitoring processes not only aid in maintaining the quality of the model but also 
serve as a foundation for continuous improvement, feeding into the model’s ongoing training and 
development to enhance it …

… last 800 chars of prompt:

…  for aligning the model’s output 
with ethical standards and societal values, thereby enhancing reliability and user trust.
Incorporating human feedback is particularly important as it provides a necessary check against the 
model’s learning biases and errors, ensuring that outputs are both technically sound and socially 
appropriate. As language models continue to evolve, these strategies ensure they remain eﬀective 
tools capable of understanding and generating human language in a manner that is both useful 
and responsible.
Summary
$is chapter provided a review of each component in the LLMOps landscape by iterating through 
data preparation, model development, governance and review, model serving, and monitoring. In the 
next chapter, we’ll dive deeper into data preparation.

FACTS:
-

Approx. tokens: 650

!!! CRITICAL: The chunk text is NOT present in the prompt → variable bug!

Do NOT proceed — fix the .format() or variable first.

In [ ]:
# ────────────────────────────────────────────────────────────────
#   FORCED RE-ASSIGNMENT + STRONG SANITY CHECK
# ────────────────────────────────────────────────────────────────

CHUNK_IDX = 40

# Force re-read from the original list — never trust previous variables
try:
    current_chunk = chunks[CHUNK_IDX]
except (NameError, IndexError) as e:
    console.print("[bold red]Cannot access chunks list:[/]", str(e))
    raise

# Print identity + length to detect overwrites
console.print(f"[yellow]Chunk object id:[/] {id(current_chunk)}")
console.print(f"[yellow]Length:[/] {len(current_chunk):,} chars")

# Very strong signature check — must contain these phrases
required_phrases = [
    "Continuous improvement",
    "catastrophic forgetting",
    "monitoring processes",
    "human feedback"
]

missing = [p for p in required_phrases if p not in current_chunk]
if missing:
    console.print("[bold red]This is NOT the expected chunk![/bold red]")
    console.print("Missing phrases:", missing)
    console.print("\nActual beginning:\n", repr(current_chunk[:300]))
    raise ValueError("Wrong chunk content — see above")

# If we reach here → content looks correct
console.print("[bold green]Chunk signature check passed ✓[/bold green]")

# ── Now safely format ──────────────────────────────────────────
facts_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=current_chunk)

# Show beginning and end to confirm
console.print("\n[magenta]Prompt preview — first 400 chars[/magenta]")
console.print(repr(facts_prompt[:400]))

console.print("\n[magenta]Prompt preview — last 600 chars[/magenta]")
console.print(repr(facts_prompt[-600:]))

# Only run model if everything looks correct
if "Continuous improvement" in facts_prompt[-2000:] and "Summary" in facts_prompt[-600:]:
    console.print("\n[green bold]→ Prompt contains correct chunk text — safe to run[/green bold]")

    facts = run_model(model_A, tokenizer_A, facts_prompt)
    pp("Extracted Facts (raw)", facts)

    # your original checks
    stripped = facts.strip()
    if "NO_FACTS" in stripped.upper() or len(stripped) < 60:
        console.print("[yellow]Warning: weak/no facts returned[/yellow]")
    else:
        console.print("[green]→ Facts look usable[/green]")
else:
    console.print("[bold red]Prompt still does not contain the chunk → bug persists[/bold red]")

Chunk object id: 1110723648

Length: 2,914 chars

Chunk signature check passed ✓

Prompt preview — first 400 chars

'\nYou are a strict information extraction system.\n\nExtract ONLY factual, domain-specific knowledge from the 
text.\n\nSTRICT RULES:\n- Output MUST be bullet points.\n- Each bullet MUST be a standalone atomic fact.\n- No 
rewriting or summarizing.\n- No interpretation.\n- No opinions.\n- No instructions.\n- No assumptions.\n- No meta 
text (chapter names, copyright, references).\n- No hallucinations.\n- If no doma'

Prompt preview — last 600 chars

'ssary check against the \nmodel’s learning biases and errors, ensuring that outputs are both technically sound and
socially \nappropriate. As language models continue to evolve, these strategies ensure they remain eﬀective \ntools
capable of understanding and generating human language in a manner that is both useful \nand 
responsible.\nSummary\n$is chapter provided a review of each component in the LLMOps landscape by iterating through
\ndata preparation, model development, governance and review, model serving, and monitoring. In the \nnext chapter,
we’ll dive deeper into data preparation.\n\nFACTS:\n- \n'

Prompt still does not contain the chunk → bug persists

In [ ]:
# ────────────────────────────────────────────────────────────────
#   Single safe cell: pick chunk → check → build prompt → run
# ────────────────────────────────────────────────────────────────

CHUNK_IDX = 40

# ── 1. Force-get the chunk from the list (never trust previous variables)
chunk_to_use = chunks[CHUNK_IDX]

# ── 2. Quick content verification
print("First 120 chars of chunk_to_use:", repr(chunk_to_use[:120]))
print("Contains 'Continuous improvement'?", "Continuous improvement" in chunk_to_use)
print("Length:", len(chunk_to_use))

# ── 3. Build prompt with EXACT same variable
facts_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=chunk_to_use)

# ── 4. Show critical parts of what will actually be sent
print("\nLast 400 characters that will be sent to the model:")
print(repr(facts_prompt[-400:]))

# Quick safety check
if "data preparation" in facts_prompt[-600:] and "Summary" in facts_prompt:
    print("\n→ Looks like the correct chunk is now in the prompt")
else:
    print("\n!!! Still wrong content in prompt – do not run model yet !!!")
    print("You must find which cell is using a different variable name.")
    raise SystemExit("Stopped – fix variable mismatch first")

# ── 5. Only if we reach here → run the model
print("\nRunning model_A …")
facts = run_model(model_A, tokenizer_A, facts_prompt)

# Show result
print("\n--- Extracted Facts ---")
print(facts)

# Your checks
stripped = facts.strip()
if "NO_FACTS" in stripped.upper() or len(stripped) < 60:
    print("⚠️ Weak or no facts returned")
else:
    print("→ Facts look usable")

First 120 chars of chunk_to_use: 'Continuous improvement\n37\nincorrect or nonsensical information – is critical to maintaining the trustworthiness and reli'
Contains 'Continuous improvement'? True
Length: 2914

Last 400 characters that will be sent to the model:
'they remain eﬀective \ntools capable of understanding and generating human language in a manner that is both useful \nand responsible.\nSummary\n$is chapter provided a review of each component in the LLMOps landscape by iterating through \ndata preparation, model development, governance and review, model serving, and monitoring. In the \nnext chapter, we’ll dive deeper into data preparation.\n\nFACTS:\n- \n'

→ Looks like the correct chunk is now in the prompt

Running model_A …

--- Extracted Facts ---
system
You are a precise and strict AI.
user

You are a strict information extraction system.

Extract ONLY factual, domain-specific knowledge from the text.

STRICT RULES:
- Output MUST be bullet points.
- Each bullet MUST be

In [ ]:
DOMAIN_EXTRACTION_PROMPT = """
You are a precise domain-knowledge extractor for continued pretraining data.

Extract factual statements from the provided text that are useful for teaching an LLM about LLM development, training, deployment, evaluation, safety, operations, scaling, or related technical/practical topics.

STRICT RULES:
- Output ONLY bullet points starting with "- "
- Each bullet must be one concise, standalone, verifiable fact
- Keep statements as close to the original wording as possible
- Do NOT interpret, generalize, combine, or add information
- Do NOT include opinions, instructions, questions, meta-commentary
- Do NOT mention chapter titles, section names, page numbers, figure references
- Do NOT output any introductory text, conclusion, or explanation
- If the text contains almost no extractable factual content (pure boilerplate, navigation, empty), output exactly one line:
NO_FACTS
- If only weak/general/high-level statements are present, still extract them — do not refuse just because they are not very specific

TEXT:
{chunk}

Output format example:
- Fact one in original-like wording
- Fact two ...
"""

In [ ]:
test_chunk = chunks[40]

# Old prompt
old_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=test_chunk)
# print(old_prompt[-300:])  # to see ending

# New prompt
new_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=test_chunk)  # ← use the updated one above

# Compare what the model would see
print("Old ending:", repr(old_prompt[-400:]))
print("New ending:", repr(new_prompt[-400:]))

Old ending: 'ing human language in a manner that is both useful \nand responsible.\nSummary\n$is chapter provided a review of each component in the LLMOps landscape by iterating through \ndata preparation, model development, governance and review, model serving, and monitoring. In the \nnext chapter, we’ll dive deeper into data preparation.\n\nOutput format example:\n- Fact one in original-like wording\n- Fact two ...\n'
New ending: 'ing human language in a manner that is both useful \nand responsible.\nSummary\n$is chapter provided a review of each component in the LLMOps landscape by iterating through \ndata preparation, model development, governance and review, model serving, and monitoring. In the \nnext chapter, we’ll dive deeper into data preparation.\n\nOutput format example:\n- Fact one in original-like wording\n- Fact two ...\n'


In [ ]:
import random

def is_promising_chunk(text: str) -> bool:
    text = text.strip()
    if len(text) < 600:                   return False   # too short
    if len(text) > 4500:                  return False   # risk of truncation
    if text.count('\n') < 8:              return False   # almost no structure
    words = len(text.split())
    if words < 100:                       return False
    # Heuristic: too code-like / table-like / symbol-heavy
    special = sum(c in "│└├═→•※★|*-=+#" for c in text) / max(1, len(text))
    if special > 0.12:                    return False
    # Bonus: contains at least some sentence-like structure
    if text.count('.') < 5:               return False
    return True


# Create candidate pool
good_chunks = [c for c in chunks if is_promising_chunk(c)]

print(f"Total chunks: {len(chunks)}")
print(f"Good candidates: {len(good_chunks)} ({len(good_chunks)/len(chunks):.1%})")

if good_chunks:
    # Pick one randomly from the filtered list
    selected_chunk = random.choice(good_chunks)
    selected_idx = chunks.index(selected_chunk)  # original position (for logging)
    print(f"Selected chunk #{selected_idx}  ({len(selected_chunk):,} chars)")
else:
    print("No good chunks found — review filter thresholds")

Total chunks: 153
Good candidates: 142 (92.8%)
Selected chunk #18  (2,327 chars)


In [ ]:
def chunk_score(text: str) -> float:
    score = 0.0

    # prefer medium-long chunks
    score += min(len(text) / 1800, 1.5)

    # reward paragraph structure
    score += (text.count('\n') / 25) * 0.4

    # reward real sentences
    score += (text.count('.') / 40) * 0.5

    # domain keyword bonus (fixed syntax)
    lower_text = text.lower()
    if any(w in lower_text for w in ["train", "data", "fine-tune", "loss", "eval", "bias"]):
        score += 0.8

    # slightly smaller bonus for related terms
    if any(w in lower_text for w in ["model", "llm", "pretrain", "inference", "token", "gradient"]):
        score += 0.6

    # penalty for link-heavy / spam-like content
    if text.count("http") > 3 or text.count("www.") > 2:
        score *= 0.4

    # never return zero or negative (for weighted sampling)
    return max(score, 0.1)

In [ ]:
# One-liner version you can put in a cell and re-run many times
import random

candidates = [i for i, c in enumerate(chunks) if 700 <= len(c) <= 3800 and c.count('\n') >= 10 and len(c.split()) >= 120]
if candidates:
    idx = random.choice(candidates)
    print(f"→ Selected chunk #{idx} ({len(chunks[idx]):,} chars)")
    # then use chunks[idx]
else:
    print("No suitable candidates — lower thresholds")

→ Selected chunk #15 (2,495 chars)


In [ ]:
import random

# ────────────────────────────────────────────────
#  Helper: decide if chunk is worth processing
# ────────────────────────────────────────────────
def is_good_chunk(text: str) -> bool:
    text = text.strip()
    if len(text) < 700:                   return False
    if len(text) > 4200:                  return False   # avoid truncation risk
    if text.count('\n') < 10:             return False
    words = len(text.split())
    if words < 110:                       return False
    if text.count('.') < 6:               return False   # too few sentences
    # avoid very table/code/symbol heavy chunks
    special_ratio = sum(c in "│└├═→•※★|*-=+#" for c in text) / max(1, len(text))
    if special_ratio > 0.11:              return False
    return True


# ────────────────────────────────────────────────
#  Level 1 style: filter → uniform random among good ones
# ────────────────────────────────────────────────
good_indices = [i for i, c in enumerate(chunks) if is_good_chunk(c)]

if not good_indices:
    print("No chunks pass the basic quality filter. Lower thresholds or check data.")
else:
    selected_idx = random.choice(good_indices)
    selected_chunk = chunks[selected_idx]

    print(f"→ Auto-selected chunk #{selected_idx}")
    print(f"  length: {len(selected_chunk):,} chars")
    print(f"  lines:  {selected_chunk.count('\n') + 1}")
    print(f"  words:  {len(selected_chunk.split())}")
    print("-" * 60)
    print(selected_chunk[:500].replace("\n", "  ").strip() + " …")
    print("-" * 60)

→ Auto-selected chunk #94
  length: 2,968 chars
  lines:  76
  words:  294
------------------------------------------------------------
Operationalizing compliance and performance  99  Furthermore, before releasing any data-driven feature or model, legal teams should be looped in via the   Airﬂow workﬂow to review the licensing and compliance posture. $is human-in-the-loop approach   ensures that legal and ethical standards are not only met but ingrained within the organization’s   operational procedures.  Let’s review an Airﬂow DAG that performs these tasks for us:  from datetime import datetime, timedelta  from airflow import DAG  from …
------------------------------------------------------------


In [ ]:
# ────────────────────────────────────────────────────────────────
#   AUTOMATIC CHUNK → PROMPT → EXTRACTION
# ────────────────────────────────────────────────────────────────

# (paste the is_good_chunk() and good_indices selection code from above here)

# If no good chunks → stop
if not good_indices:
    raise ValueError("No suitable chunks found — check filtering logic")

# ── Use the auto-selected chunk ────────────────────────────────
current_chunk = selected_chunk   # already selected above

# Optional: very light content check (not mandatory anymore)
if len(current_chunk) < 600 or "http" in current_chunk[:300]:
    print("[yellow]Note: selected chunk is short or link-heavy[/yellow]")

# ── Build prompt ───────────────────────────────────────────────
try:
    facts_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=current_chunk)
except KeyError as e:
    print(f"[red]Prompt formatting failed — missing placeholder {e}[/red]")
    raise

# ── Optional: quick sanity print (helpful during development)
print("[dim]Prompt ending preview (last ~300 chars):[/dim]")
print(repr(facts_prompt[-300:]))
print("-" * 70)

# ── Run extraction ─────────────────────────────────────────────
print("[bold cyan]Running fact extraction...[/bold cyan]")
facts = run_model(model_A, tokenizer_A, facts_prompt)

# ── Show & check result ────────────────────────────────────────
pp("Extracted Facts (raw)", facts)

stripped = facts.strip()
if "NO_FACTS" in stripped.upper() or len(stripped) < 60 or stripped.count("- ") < 1:
    print("[yellow]Warning: weak / no facts returned — consider softer prompt or skip[/yellow]")
else:
    print("[green bold]→ Facts look usable ✓[/green bold]")

[dim]Prompt ending preview (last ~300 chars):[/dim]
' the preceding, we’re de!ning our operators for the calculations, checks, and human intervention \nstages. Lastly, in the following, we de!ne the serial execution of the steps:\ncalculate_metrics >> check_metrics >> send_email\n\nOutput format example:\n- Fact one in original-like wording\n- Fact two ...\n'
----------------------------------------------------------------------
[bold cyan]Running fact extraction...[/bold cyan]


╭────────────────────────────────────────────  Extracted Facts (raw)  ────────────────────────────────────────────╮
│                                                                                                                 │
│  system                                                                                                         │
│  You are a precise and strict AI.                                                                               │
│  user                                                                                                           │
│                                                                                                                 │
│  You are a precise domain-knowledge extractor for continued pretraining data.                                   │
│                                                                                                                 │
│  Extract factual statements from the provided text that are useful for teaching an LLM about LLM development,   │
│  training, deployment, evaluation, safety, operations, scaling, or related technical/practical topics.          │
│                                                                                                                 │
│  STRICT RULES:                                                                                                  │
│  - Output ONLY bullet points starting with "- "                                                                 │
│  - Each bullet must be one concise, standalone, verifiable fact                                                 │
│  - Keep statements as close to the original wording as possible                                                 │
│  - Do NOT interpret, generalize, combine, or add information                                                    │
│  - Do NOT include opinions, instructions, questions, meta-commentary                                            │
│  - Do NOT mention chapter titles, section names, page numbers, figure references                                │
│  - Do NOT output any introductory text, conclusion, or explanation                                              │
│  - If the text contains almost no extractable factual content (pure boilerplate, navigation, empty), output     │
│  exactly one line:                                                                                              │
│  NO_FACTS                                                                                                       │
│  - If only weak/general/high-level statements are present, still extract them — do not refuse just because      │
│  they are not very specific                                                                                     │
│                                                                                                                 │
│  TEXT:                                                                                                          │
│  Operationalizing compliance and performance                                                                    │
│  99                                                                                                             │
│  Furthermore, before releasing any data-driven feature or model, legal teams should be looped in via the        │
│  Airﬂow workﬂow to review the licensing and compliance posture. $is human-in-the-loop approach                  │
│  ensures that legal and ethical standards are not only met but ingrained within the organization’s              │
│  operational procedures.                                                                                        │
│  Let’s review an Airﬂow DAG that performs these tasks for us:                                                   │
│  from datetime import datetime, timedelta                                                                       │
│  from airflow import DAG                              

[yellow]Warning: weak / no facts returned — consider softer prompt or skip[/yellow]


In [ ]:
def chunk_score(text: str) -> float:
    score = 0.0
    lt = text.lower()
    score += min(len(text) / 1800.0, 1.5)
    score += (text.count('\n') / 25.0) * 0.4
    score += (text.count('.') / 40.0) * 0.5
    if any(w in lt for w in ["train","data","fine-tune","loss","eval","bias","llm","pretrain","inference","token","gradient","catastrophic forgetting"]):
        score += 0.9
    if text.count("http") > 3 or text.count("www.") > 2:
        score *= 0.35
    return max(score, 0.1)

# Weighted selection
scores = [chunk_score(c) for c in chunks]
total_score = sum(scores)
probs = [s / total_score for s in scores]

selected_idx = random.choices(range(len(chunks)), weights=probs, k=1)[0]
selected_chunk = chunks[selected_idx]

print(f"→ Weighted-selected chunk #{selected_idx}  (score ≈ {chunk_score(selected_chunk):.2f})")

→ Weighted-selected chunk #11  (score ≈ 3.25)


In [ ]:
DOMAIN_EXTRACTION_PROMPT = """
You are a precise domain-knowledge extractor. Your ONLY job is to output bullet points — nothing else.

STRICT RULES — YOU MUST FOLLOW ALL OF THEM:
- Output ONLY lines that start with "- "
- Each line must be ONE concise, standalone, verbatim-style fact from the text
- Keep wording extremely close to the original — do not paraphrase unless absolutely necessary for clarity
- NO introductory sentences, NO conclusions, NO explanations, NO "here are", NO "examples", NO "Fact:" prefix
- NO opinions, instructions, questions, meta text, chapter/page references
- If almost nothing extractable → output exactly: NO_FACTS
- Even for high-level or code-related text, extract correct technical statements

Do NOT add any text before or after the bullet list.

TEXT:
{chunk}
"""

In [ ]:
import json
import random
import time

# ============== YOUR PROMPTS (improved & strict) ==============
DOMAIN_EXTRACTION_PROMPT = """
You are a precise domain-knowledge extractor. Your ONLY job is to output bullet points — nothing else.

STRICT RULES — FOLLOW ALL:
- Output ONLY lines that start with "- "
- Each line = one concise fact, keep original wording as much as possible
- NO intro, NO explanation, NO "Here are", NO "Fact:", NO extra text
- If almost nothing useful → output exactly: NO_FACTS

TEXT:
{chunk}
"""

IFT_GENERATION_PROMPT = """
You are generating high-quality instruction fine-tuning data.
Generate EXACTLY ONE JSON object. No extra text.

FACTS:
{facts}

Output ONLY this exact format:
{{
  "instruction": "...",
  "input": "...",
  "response": "..."
}}
"""

SELF_EVAL_PROMPT = """
You are a strict validator.
Sample:
{sample}

Output ONLY one word: accept or reject
"""

# ============== SIMPLE CHUNK FILTER ==============
def is_good_chunk(text):
    text = text.strip()
    if len(text) < 700: return False
    if len(text) > 4200: return False
    if text.count('\n') < 8: return False
    if len(text.split()) < 110: return False
    if text.count('.') < 5: return False
    return True

# ============== FIND GOOD CHUNKS ==============
good_indices = [i for i, c in enumerate(chunks) if is_good_chunk(c)]
print(f"Found {len(good_indices)} good chunks out of {len(chunks)} total\n")

# ============== COLLECT SAMPLES ==============
dataset = []          # will hold all accepted samples
target = 250          # change to 150 or 300 if you want
max_attempts = 400    # safety limit

print("=== Starting IFT dataset creation ===\n")

for attempt in range(max_attempts):
    if len(dataset) >= target:
        print(f"\nReached target of {target} samples! Stopping.")
        break

    if not good_indices:
        print("No more good chunks left.")
        break

    idx = random.choice(good_indices)
    good_indices.remove(idx)          # don't reuse same chunk
    chunk = chunks[idx]

    print(f"[{len(dataset)+1}/{target}] Trying chunk #{idx} ({len(chunk)} chars)... ", end="")

    # 1. Extract facts
    facts_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=chunk)
    facts = run_model(model_A, tokenizer_A, facts_prompt)

    facts = facts.strip()
    if "NO_FACTS" in facts.upper() or len(facts) < 80 or "- " not in facts:
        print("no facts → skip")
        continue

    # 2. Generate IFT sample
    ift_prompt = IFT_GENERATION_PROMPT.format(facts=facts)
    raw_sample = run_model(model_B, tokenizer_B, ift_prompt, max_new_tokens=400)

    sample_json = extract_json(raw_sample)   # your existing function

    if sample_json is None:
        print("bad JSON → skip")
        continue

    # 3. Self-evaluation
    eval_prompt = SELF_EVAL_PROMPT.format(sample=json.dumps(sample_json))
    verdict = run_model(model_B, tokenizer_B, eval_prompt, max_new_tokens=20, temperature=0.0)

    if "accept" in verdict.lower():
        dataset.append(sample_json)
        print("ACCEPTED ✓")

        # Save immediately (so you never lose progress)
        with open("llmops_ift_dataset.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(sample_json, ensure_ascii=False) + "\n")
    else:
        print("rejected")

# ============== FINAL SUMMARY ==============
print("\n" + "="*60)
print(f"DONE! You now have {len(dataset)} accepted IFT samples")
print(f"Saved to: llmops_ift_dataset.jsonl")
print("="*60)

# Show first sample as example
if dataset:
    print("\nFirst sample example:")
    print(json.dumps(dataset[0], indent=2))

Found 146 good chunks out of 153 total

=== Starting IFT dataset creation ===

[1/250] Trying chunk #20 (2681 chars)... no facts → skip
[1/250] Trying chunk #138 (2999 chars)... no facts → skip
[1/250] Trying chunk #66 (1520 chars)... no facts → skip
[1/250] Trying chunk #109 (3214 chars)... no facts → skip
[1/250] Trying chunk #143 (3393 chars)... no facts → skip
[1/250] Trying chunk #50 (2275 chars)... no facts → skip
[1/250] Trying chunk #81 (1503 chars)... no facts → skip
[1/250] Trying chunk #80 (2982 chars)... no facts → skip
[1/250] Trying chunk #29 (2103 chars)... no facts → skip
[1/250] Trying chunk #106 (2206 chars)... no facts → skip
[1/250] Trying chunk #90 (2424 chars)... no facts → skip
[1/250] Trying chunk #110 (2510 chars)... no facts → skip
[1/250] Trying chunk #8 (3411 chars)... no facts → skip
[1/250] Trying chunk #49 (1654 chars)... no facts → skip
[1/250] Trying chunk #113 (3329 chars)... no facts → skip
[1/250] Trying chunk #83 (3354 chars)... no facts → skip
[1/2

KeyboardInterrupt: 

In [ ]:
import json
import random

# ────────────────────────────────────────────────
#   VERY SOFT FACT EXTRACTION PROMPT
# ────────────────────────────────────────────────
DOMAIN_EXTRACTION_PROMPT = """
Extract as many factual statements as possible from the text.
Output only bullet points starting with "- ".
Each bullet = one short fact.
Keep wording close to the original.
If absolutely nothing useful → write only: NO_FACTS
Otherwise always try to output at least 1–5 bullets even if they are general or high-level.

TEXT:
{chunk}
"""

IFT_GENERATION_PROMPT = """
You are generating high-quality instruction fine-tuning data.
Generate EXACTLY ONE JSON object. No extra text.

FACTS:
{facts}

Output ONLY this exact format:
{{
  "instruction": "...",
  "input": "...",
  "response": "..."
}}
"""

SELF_EVAL_PROMPT = """
You are a strict validator.
Sample:
{sample}

Output ONLY one word: accept or reject
"""

# ────────────────────────────────────────────────
#   SIMPLE CHUNK FILTER
# ────────────────────────────────────────────────
def is_good_chunk(text):
    text = text.strip()
    if len(text) < 600: return False
    if len(text) > 4500: return False
    if text.count('\n') < 6: return False
    if len(text.split()) < 90: return False
    return True

# Find good chunks
good_indices = [i for i, c in enumerate(chunks) if is_good_chunk(c)]
print(f"Found {len(good_indices)} good chunks out of {len(chunks)} total\n")

# ────────────────────────────────────────────────
#   MAIN LOOP — CREATE IFT SAMPLES
# ────────────────────────────────────────────────
dataset = []                # collected good samples
target_samples = 20         # change to 50, 100, 200... later
max_attempts = 60           # safety limit — prevents infinite loop

print("=== Starting IFT dataset creation ===\n")

for attempt in range(1, max_attempts + 1):
    if len(dataset) >= target_samples:
        print(f"\nReached target of {target_samples} good samples!")
        break

    if not good_indices:
        print("No more good chunks left.")
        break

    idx = random.choice(good_indices)
    good_indices.remove(idx)  # avoid reusing same chunk
    chunk_text = chunks[idx]

    print(f"[{len(dataset)+1}/{target_samples}] Chunk #{idx} ({len(chunk_text)} chars) → ", end="")

    # 1. Extract facts
    facts_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=chunk_text)
    raw_facts = run_model(model_A, tokenizer_A, facts_prompt)

    # Show what we actually got
    print("facts preview:", raw_facts[:120].replace("\n", " ").strip() + "...")

    # Clean facts a little
    lines = [line.strip() for line in raw_facts.split('\n') if line.strip()]
    bullet_lines = [line for line in lines if line.startswith('- ')]

    if bullet_lines:
        facts = "\n".join(bullet_lines)
    elif "NO_FACTS" in raw_facts.upper():
        facts = "NO_FACTS"
    else:
        facts = raw_facts.strip()  # fallback

    # Accept even short facts for now
    if facts == "NO_FACTS" or len(facts.strip()) < 30:
        print("→ no / too weak facts → skip")
        continue

    # 2. Generate one IFT sample
    ift_prompt = IFT_GENERATION_PROMPT.format(facts=facts)
    raw_json = run_model(model_B, tokenizer_B, ift_prompt, max_new_tokens=450)

    sample_dict = extract_json(raw_json)

    if sample_dict is None:
        print("→ JSON parse failed → skip")
        continue

    # 3. Self-evaluation
    eval_prompt = SELF_EVAL_PROMPT.format(sample=json.dumps(sample_dict))
    verdict = run_model(model_B, tokenizer_B, eval_prompt, max_new_tokens=30, temperature=0.0)

    if "accept" in verdict.lower():
        dataset.append(sample_dict)
        print("→ ACCEPTED ✓")

        # Save immediately (append mode)
        with open("my_first_ift_dataset.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(sample_dict, ensure_ascii=False) + "\n")
    else:
        print("→ rejected")

# ────────────────────────────────────────────────
#   FINAL SUMMARY
# ────────────────────────────────────────────────
print("\n" + "="*70)
print(f"Finished! You collected {len(dataset)} accepted samples")
print(f"Saved to: my_first_ift_dataset.jsonl")
print("="*70)

if dataset:
    print("\nExample of first accepted sample:")
    print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

Found 153 good chunks out of 153 total

=== Starting IFT dataset creation ===

[1/20] Chunk #56 (2453 chars) → 

facts preview: system You are a precise and strict AI. user  Extract as many factual statements as possible from the text. Output only...
→ JSON parse failed → skip
[1/20] Chunk #116 (3260 chars) → facts preview: system You are a precise and strict AI. user  Extract as many factual statements as possible from the text. Output only...
→ JSON parse failed → skip
[1/20] Chunk #59 (1887 chars) → facts preview: system You are a precise and strict AI. user  Extract as many factual statements as possible from the text. Output only...
→ no / too weak facts → skip
[1/20] Chunk #106 (2206 chars) → facts preview: system You are a precise and strict AI. user  Extract as many factual statements as possible from the text. Output only...
→ no / too weak facts → skip
[1/20] Chunk #94 (2968 chars) → facts preview: system You are a precise and strict AI. user  Extract as many factual statements as possible from the text. Output only...
→ JSON parse failed → skip
[1/20] Chunk #18 (2327 chars) → facts pre

KeyboardInterrupt: 

In [ ]:
test_chunk = chunks[40]  # or any index
test_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=test_chunk[:2000])  # limit length for test

print("Prompt sent:")
print(test_prompt[:400], "...\n")

raw_output = run_model(model_A, tokenizer_A, test_prompt, max_new_tokens=200)

print("Raw model output:")
print(raw_output)
print("\nLength of output:", len(raw_output))

Prompt sent:

Extract as many factual statements as possible from the text.
Output only bullet points starting with "- ".
Each bullet = one short fact.
Keep wording close to the original.
If absolutely nothing useful → write only: NO_FACTS
Otherwise always try to output at least 1–5 bullets even if they are general or high-level.

TEXT:
Continuous improvement
37
incorrect or nonsensical information – is critic ...

Raw model output:
system
You are a precise and strict AI.
user

Extract as many factual statements as possible from the text.
Output only bullet points starting with "- ".
Each bullet = one short fact.
Keep wording close to the original.
If absolutely nothing useful → write only: NO_FACTS
Otherwise always try to output at least 1–5 bullets even if they are general or high-level.

TEXT:
Continuous improvement
37
incorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of 
the LLM. $ese monitoring processes not only aid in maintaini

In [ ]:
def run_model(model, tokenizer, prompt, max_new_tokens=300, temperature=0.7, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    print("Input token count:", inputs.input_ids.shape[1])   # debug

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=(temperature > 0.01),
        pad_token_id=tokenizer.eos_token_id,
        **kwargs
    )

    # Slice only new tokens
    new_tokens = outputs[0, inputs.input_ids.shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print("Generated token count:", len(new_tokens))   # debug
    return decoded

In [ ]:
test_chunk = chunks[40]
test_prompt = DOMAIN_EXTRACTION_PROMPT.format(chunk=test_chunk[:2500])

raw_output = run_model(model_A, tokenizer_A, test_prompt, max_new_tokens=300, temperature=0.1)

print("Clean generated output:")
print(raw_output)
print("\nLength:", len(raw_output))

Input token count: 538
Generated token count: 300
Clean generated output:
Extract as many factual statements as possible from the text.
Output only bullet points starting with "- ".
Each bullet = one short fact.
Keep wording close to the original.
If absolutely nothing useful → write only: NO_FACTS
Otherwise always try to output at least 1–5 bullets even if they are general or high-level.

TEXT:
Continuous improvement
37
incorrect or nonsensical information – is critical to maintaining the trustworthiness and reliability of 
the LLM. $ese monitoring processes not only aid in maintaining the quality of the model but also 
serve as a foundation for continuous improvement, feeding into the model’s ongoing training and 
development to enhance its performance over time.
Continuous improvement
In the realm of LLMs such as GPT, continuous improvement is paramount to maintaining their 
relevance and e%cacy. $is process is multifaceted and involves integrating new data, developing 
strategies t

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Choose one of these two lines — depending on what you want to test

# Option 1: Use the instruct-tuned version (strongly recommended)
model_name = "Qwen/Qwen2-1.5B-Instruct"

# Option 2: If you want to stay with the base model for some reason
# model_name = "Qwen/Qwen2-1.5B"

print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model — use float16 / bfloat16 if you have GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",               # auto = cuda if available, else cpu
    trust_remote_code=True
)

print("Model and tokenizer loaded successfully.")

Loading model: Qwen/Qwen2-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model and tokenizer loaded successfully.


In [17]:
chunk_text = chunks[40][:2000]  # or any other chunk

messages = [
    {"role": "system", "content": "You are a precise domain-knowledge extractor. Output ONLY bullet points starting with '- '. Each bullet one short fact. No extra text, no explanations, no introductions."},
    {"role": "user", "content": f"Extract as many factual statements as possible from this text.\n\nTEXT:\n{chunk_text}"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    do_sample=False
)

new_tokens = outputs[0][inputs.input_ids.shape[1]:]
generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Generated output:")
print(generated)

Generated output:
- Continuous improvement is critical to maintaining the trustworthiness and reliability of LLMs.
- Monitoring processes aid in maintaining the quality of the model and serve as a foundation for continuous improvement.
- Integrating new data ensures that models grasp current terminologies, phrases, and relevant topics.
- Incorporating human feedback helps maintain the model's overall performance across a broad spectrum of topics.
- Techniques like Elastic Weight Consolidation and Progressive Neural Networks are used to mitigate catastrophic forgetting.
- Human feedback is crucial for ensuring that the model's responses are accurate and contextually and emotionally congruent with human expectations.


In [18]:
import json
import random

# ────────────────────────────────────────────────
#   FACT EXTRACTION — NOW USING CHAT TEMPLATE
# ────────────────────────────────────────────────
# New: system prompt is now separate, using apply_chat_template
SYSTEM_PROMPT_FACTS = (
    "You are a precise domain-knowledge extractor. "
    "Output ONLY bullet points starting with '- '. "
    "Each bullet must be one concise, standalone fact. "
    "Use wording close to the original text. "
    "No introductions, no explanations, no numbering, no extra text at all."
)

# ────────────────────────────────────────────────
#   IFT & EVAL PROMPTS (unchanged except escaped braces)
# ────────────────────────────────────────────────
IFT_GENERATION_PROMPT = """
You are generating high-quality instruction fine-tuning data.
Generate EXACTLY ONE JSON object. No extra text.

FACTS:
{facts}

Output ONLY this exact format:
{{
  "instruction": "...",
  "input": "...",
  "response": "..."
}}
"""

SELF_EVAL_PROMPT = """
You are a strict validator.
Sample:
{sample}

Output ONLY one word: accept or reject
"""

# ────────────────────────────────────────────────
#   SIMPLE CHUNK FILTER (slightly relaxed)
# ────────────────────────────────────────────────
def is_good_chunk(text):
    text = text.strip()
    if len(text) < 600: return False
    if len(text) > 4500: return False
    if text.count('\n') < 6: return False
    if len(text.split()) < 90: return False
    return True

# Find good chunks
good_indices = [i for i, c in enumerate(chunks) if is_good_chunk(c)]
print(f"Found {len(good_indices)} good chunks out of {len(chunks)} total\n")

# ────────────────────────────────────────────────
#   MAIN LOOP — CREATE IFT SAMPLES
# ────────────────────────────────────────────────
dataset = []                # collected good samples
target_samples = 20         # ← you can increase later (50, 100, 200...)
max_attempts = 80           # safety limit

print("=== Starting IFT dataset creation ===\n")

for attempt in range(1, max_attempts + 1):
    if len(dataset) >= target_samples:
        print(f"\nReached target of {target_samples} good samples!")
        break

    if not good_indices:
        print("No more good chunks left.")
        break

    idx = random.choice(good_indices)
    good_indices.remove(idx)
    chunk_text = chunks[idx]

    print(f"[{len(dataset)+1}/{target_samples}] Chunk #{idx} ({len(chunk_text)} chars) → ", end="")

    # ── 1. Extract facts using chat template  ──────────────────────
    #           ← NEW: using messages + apply_chat_template
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FACTS},
        {"role": "user",   "content": f"Extract as many factual statements as possible from this text.\n\nTEXT:\n{chunk_text}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    raw_facts = run_model(
        model, tokenizer, prompt,
        max_new_tokens=400,
        temperature=0.1,
        do_sample=False
    )

    # Show preview
    print("facts preview:", raw_facts[:120].replace("\n", " ").strip() + "...")

    # Clean → keep only real bullet lines
    lines = [line.strip() for line in raw_facts.split('\n') if line.strip()]
    bullet_lines = [line for line in lines if line.startswith('- ')]

    if bullet_lines:
        facts = "\n".join(bullet_lines)
    else:
        facts = ""

    # Lowered threshold so it's easier to pass in early testing
    if not bullet_lines or len(bullet_lines) < 1:
        print("→ too few / no bullets → skip")
        continue

    # ── 2. Generate one IFT sample ────────────────────────────────
    ift_prompt = IFT_GENERATION_PROMPT.format(facts=facts)
    raw_json = run_model(
        model, tokenizer, ift_prompt,
        max_new_tokens=450,
        temperature=0.4
    )

    sample_dict = extract_json(raw_json)

    if sample_dict is None:
        print("→ JSON parse failed → skip")
        continue

    # ── 3. Self-evaluation ────────────────────────────────────────
    eval_prompt = SELF_EVAL_PROMPT.format(sample=json.dumps(sample_dict))
    verdict = run_model(
        model, tokenizer, eval_prompt,
        max_new_tokens=30,
        temperature=0.0
    )

    if "accept" in verdict.lower():
        dataset.append(sample_dict)
        print("→ ACCEPTED ✓")

        # Save immediately (append mode)
        with open("my_first_ift_dataset.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(sample_dict, ensure_ascii=False) + "\n")
    else:
        print("→ rejected")

# ────────────────────────────────────────────────
#   FINAL SUMMARY
# ────────────────────────────────────────────────
print("\n" + "="*70)
print(f"Finished! Collected {len(dataset)} accepted samples")
print(f"Saved to: my_first_ift_dataset.jsonl")
print("="*70)

if dataset:
    print("\nFirst accepted sample:")
    print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

Found 153 good chunks out of 153 total

=== Starting IFT dataset creation ===

[1/20] Chunk #139 (2931 chars) → facts preview: - Privacy and data security are critical challenges in the training and deployment of LLMs. - Techniques such as differe...
No JSON object found in output
→ JSON parse failed → skip
[1/20] Chunk #74 (1793 chars) → facts preview: - Script fetches question and context data from our Feast feature store. - Uses the !ne-tuned T5 model from a Hugging Fa...


KeyboardInterrupt: 

In [21]:
import json
import random

# ────────────────────────────────────────────────
#   FACT EXTRACTION — CHAT TEMPLATE STYLE
# ────────────────────────────────────────────────
SYSTEM_PROMPT_FACTS = (
    "You are a precise domain-knowledge extractor. "
    "Output ONLY bullet points starting with '- '. "
    "Each bullet must be one concise, standalone fact. "
    "Use wording close to the original text. "
    "No introductions, no explanations, no numbering, no extra text at all."
)

# ────────────────────────────────────────────────
#   IFT & EVAL PROMPTS (braces escaped)
# ────────────────────────────────────────────────
IFT_GENERATION_PROMPT = """
You are generating high-quality instruction fine-tuning data.
Generate EXACTLY ONE JSON object. No extra text.

FACTS:
{facts}

Output ONLY this exact format:
{{
  "instruction": "...",
  "input": "...",
  "response": "..."
}}
"""

SELF_EVAL_PROMPT = """
You are a strict validator.
Sample:
{sample}

Output ONLY one word: accept or reject
"""

# ────────────────────────────────────────────────
#   UPDATED run_model FUNCTION (no conflicting do_sample)
# ────────────────────────────────────────────────
def run_model(model, tokenizer, prompt, max_new_tokens=300, temperature=0.7, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
        **kwargs
    )

    # Only take newly generated tokens
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded

# ────────────────────────────────────────────────
#   SIMPLE CHUNK FILTER
# ────────────────────────────────────────────────
def is_good_chunk(text):
    text = text.strip()
    if len(text) < 600: return False
    if len(text) > 4500: return False
    if text.count('\n') < 6: return False
    if len(text.split()) < 90: return False
    return True

# Find good chunks
good_indices = [i for i, c in enumerate(chunks) if is_good_chunk(c)]
print(f"Found {len(good_indices)} good chunks out of {len(chunks)} total\n")

# ────────────────────────────────────────────────
#   MAIN LOOP — CREATE IFT SAMPLES
# ────────────────────────────────────────────────
dataset = []
target_samples = 100          # ← change to 50, 100, 200... later
max_attempts = 80            # safety limit

print("=== Starting IFT dataset creation ===\n")

for attempt in range(1, max_attempts + 1):
    if len(dataset) >= target_samples:
        print(f"\nReached target of {target_samples} good samples!")
        break

    if not good_indices:
        print("No more good chunks left.")
        break

    idx = random.choice(good_indices)
    good_indices.remove(idx)
    chunk_text = chunks[idx]

    print(f"[{len(dataset)+1}/{target_samples}] Chunk #{idx} ({len(chunk_text)} chars) → ", end="")

    # ── 1. Extract facts (chat format) ─────────────────────────────
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FACTS},
        {"role": "user",   "content": f"Extract as many factual statements as possible from this text.\n\nTEXT:\n{chunk_text}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    raw_facts = run_model(
        model, tokenizer, prompt,
        max_new_tokens=400,
        temperature=0.1,
        do_sample=False             # greedy → strict & reproducible
    )

    print("facts preview:", raw_facts[:120].replace("\n", " ").strip() + "...")

    # Clean → keep only real bullet lines
    lines = [line.strip() for line in raw_facts.split('\n') if line.strip()]
    bullet_lines = [line for line in lines if line.startswith('- ')]

    if not bullet_lines:
        print("→ no bullets → skip")
        continue

    facts = "\n".join(bullet_lines)

    # ── 2. Generate one IFT sample ─────────────────────────────────
    ift_prompt = IFT_GENERATION_PROMPT.format(facts=facts)
    raw_json = run_model(
        model, tokenizer, ift_prompt,
        max_new_tokens=450,
        temperature=0.7,
        do_sample=True              # some diversity for IFT
    )

    sample_dict = extract_json(raw_json)

    if sample_dict is None:
        print("→ JSON parse failed → skip")
        continue

    # ── 3. Self-evaluation ─────────────────────────────────────────
    eval_prompt = SELF_EVAL_PROMPT.format(sample=json.dumps(sample_dict))
    verdict = run_model(
        model, tokenizer, eval_prompt,
        max_new_tokens=30,
        temperature=0.0,
        do_sample=False
    )

    if "accept" in verdict.lower():
        dataset.append(sample_dict)
        print("→ ACCEPTED ✓")

        # Save immediately
        with open("my_first_ift_dataset.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(sample_dict, ensure_ascii=False) + "\n")
    else:
        print("→ rejected")

# ────────────────────────────────────────────────
#   FINAL SUMMARY
# ────────────────────────────────────────────────
print("\n" + "="*70)
print(f"Finished! Collected {len(dataset)} accepted samples")
print(f"Saved to: my_first_ift_dataset.jsonl")
print("="*70)

if dataset:
    print("\nFirst accepted sample:")
    print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

Found 153 good chunks out of 153 total

=== Starting IFT dataset creation ===

[1/100] Chunk #31 (2601 chars) → facts preview: - Reviewing LLMOps Components: A balance needs to be struck between context richness and computational efficiency when c...
No JSON object found in output
→ JSON parse failed → skip
[1/100] Chunk #95 (2537 chars) → facts preview: - The book covers advanced strategies and future directions for LLMOps, including deployment, performance, and scalabili...
JSON decode failed: Extra data: line 1 column 204 (char 203) → raw: {"instruction": "What chapters does the book cover?", "input": "", "response": "[\"Chapter 1\", \"Chapter 2\", \"Chapter 3\", \"Chapter 4\", \"Chapter 5\", \"Chapter 6\", \"Chapter 7\", \"Chapter 8\"]...
→ JSON parse failed → skip
[1/100] Chunk #100 (3637 chars) → facts preview: - LLMOps Strategies for Inference, Serving, and Scalability   - Mitigating accuracy risks when applying model pruning an...
→ ACCEPTED ✓
[2/100] Chunk #46 (1618 chars) → f

In [22]:
import json
import random
import os

# ────────────────────────────────────────────────
#   FACT EXTRACTION — CHAT TEMPLATE STYLE
# ────────────────────────────────────────────────
SYSTEM_PROMPT_FACTS = (
    "You are a precise domain-knowledge extractor. "
    "Output ONLY bullet points starting with '- '. "
    "Each bullet must be one concise, standalone fact. "
    "Use wording close to the original text. "
    "No introductions, no explanations, no numbering, no extra text at all."
)

# ────────────────────────────────────────────────
#   IFT & EVAL PROMPTS
# ────────────────────────────────────────────────
IFT_GENERATION_PROMPT = """
You are generating high-quality instruction fine-tuning data.
Generate EXACTLY ONE JSON object — no extra text, no markdown, no explanations, no ```json markers.

FACTS:
{facts}

Output exactly this structure and nothing else:
{
  "instruction": "a clear, specific question or task",
  "input": "optional context or empty string \"\"",
  "response": "accurate answer using only information from the facts"
}
"""

SELF_EVAL_PROMPT = """
You are a strict validator.
Sample:
{sample}

Output ONLY one word: accept or reject
"""

# ────────────────────────────────────────────────
#   run_model (unchanged from last working version)
# ────────────────────────────────────────────────
def run_model(model, tokenizer, prompt, max_new_tokens=300, temperature=0.7, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
        **kwargs
    )
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# ────────────────────────────────────────────────
#   extract_json (current version you are using)
# ────────────────────────────────────────────────
def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```json\s*|\s*```$', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'^(Here\'s|The function has been|formatted data:|JSON:).*?\{', '{', text, flags=re.IGNORECASE | re.DOTALL)

    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            print(f"JSON decode failed: {e} → raw: {json_str[:200]}...")
            return None

    print("No JSON object found in output")
    return None

# ────────────────────────────────────────────────
#   CHUNK FILTER
# ────────────────────────────────────────────────
def is_good_chunk(text):
    text = text.strip()
    if len(text) < 600: return False
    if len(text) > 4500: return False
    if text.count('\n') < 6: return False
    if len(text.split()) < 90: return False
    return True

good_indices = [i for i, c in enumerate(chunks) if is_good_chunk(c)]
print(f"Found {len(good_indices)} good chunks out of {len(chunks)} total\n")

# ────────────────────────────────────────────────
#   MAIN LOOP — with checkpoint every 10 samples
# ────────────────────────────────────────────────
dataset = []
target_samples = 100
max_attempts = 400           # increased safety margin
checkpoint_interval = 10     # save every 10 accepted samples

output_file = "my_first_ift_dataset.jsonl"

# If file already exists → load existing data (optional resume)
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        dataset = [json.loads(line.strip()) for line in f if line.strip()]
    print(f"Loaded {len(dataset)} existing samples from {output_file}\n")

print("=== Starting / continuing IFT dataset creation ===\n")

attempt = 1
while len(dataset) < target_samples and attempt <= max_attempts:
    if not good_indices:
        print("No more good chunks left.")
        break

    idx = random.choice(good_indices)
    good_indices.remove(idx)
    chunk_text = chunks[idx]

    print(f"[{len(dataset)+1}/{target_samples}] Chunk #{idx} ({len(chunk_text)} chars) → ", end="")

    # 1. Facts extraction
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FACTS},
        {"role": "user",   "content": f"Extract as many factual statements as possible from this text.\n\nTEXT:\n{chunk_text}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    raw_facts = run_model(model, tokenizer, prompt, max_new_tokens=400, temperature=0.1, do_sample=False)

    lines = [l.strip() for l in raw_facts.split('\n') if l.strip()]
    bullet_lines = [l for l in lines if l.startswith('- ')]

    if not bullet_lines:
        print("no bullets → skip")
        attempt += 1
        continue

    facts = "\n".join(bullet_lines)
    print("facts preview:", facts[:100].replace("\n", " ") + "...")

    # 2. IFT generation
    ift_prompt = IFT_GENERATION_PROMPT.format(facts=facts)
    raw_json_text = run_model(model, tokenizer, ift_prompt, max_new_tokens=350, temperature=0.3, do_sample=True)

    print("IFT raw:", raw_json_text[:300].replace("\n", " ").strip() + "...")

    sample_dict = extract_json(raw_json_text)

    if sample_dict is None:
        print("→ JSON parse failed → skip")
        attempt += 1
        continue

    # 3. Self-evaluation (optional — currently disabled)
    # Uncomment when you want to use it again
    """
    eval_prompt = SELF_EVAL_PROMPT.format(sample=json.dumps(sample_dict))
    verdict = run_model(model, tokenizer, eval_prompt, max_new_tokens=30, temperature=0.0, do_sample=False)
    if "accept" not in verdict.lower():
        print("→ rejected")
        attempt += 1
        continue
    """

    # Accept
    dataset.append(sample_dict)
    print("→ ACCEPTED ✓")

    # Immediate save after every sample (minimum loss protection)
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(sample_dict, ensure_ascii=False) + "\n")

    # Checkpoint message every 10 samples
    if len(dataset) % checkpoint_interval == 0:
        print(f"\n*** CHECKPOINT *** Saved {len(dataset)} samples so far\n")

    attempt += 1

# ────────────────────────────────────────────────
#   FINAL SUMMARY
# ────────────────────────────────────────────────
print("\n" + "="*70)
print(f"Finished — total collected: {len(dataset)} samples")
print(f"File: {output_file}")
print("="*70)

if dataset:
    print("\nLast accepted sample:")
    print(json.dumps(dataset[-1], indent=2, ensure_ascii=False))

Found 153 good chunks out of 153 total

Loaded 47 existing samples from my_first_ift_dataset.jsonl

=== Starting / continuing IFT dataset creation ===

[48/100] Chunk #131 (2970 chars) → facts preview: - Scaling models requires exponentially more computational power and data, which can be costly and r...


KeyError: '\n  "instruction"'

In [23]:
import json
import random
import os
import re

# ────────────────────────────────────────────────
#   FACT EXTRACTION SYSTEM PROMPT
# ────────────────────────────────────────────────
SYSTEM_PROMPT_FACTS = (
    "You are a precise domain-knowledge extractor. "
    "Output ONLY bullet points starting with '- '. "
    "Each bullet must be one concise, standalone fact. "
    "Use wording close to the original text. "
    "No introductions, no explanations, no numbering, no extra text at all."
)

# ────────────────────────────────────────────────
#   IFT GENERATION PROMPT — ESCAPED BRACES (this fixes the KeyError)
# ────────────────────────────────────────────────
IFT_GENERATION_PROMPT = """
You are generating high-quality instruction fine-tuning data.
Generate EXACTLY ONE JSON object — no extra text, no markdown, no explanations, no ```json markers.

FACTS:
{facts}

Output exactly this structure and nothing else:
{{
  "instruction": "a clear, specific question or task",
  "input": "optional context or empty string \"\"",
  "response": "accurate answer using only information from the facts"
}}
"""

SELF_EVAL_PROMPT = """
You are a strict validator.
Sample:
{sample}

Output ONLY one word: accept or reject
"""

# ────────────────────────────────────────────────
#   run_model (unchanged)
# ────────────────────────────────────────────────
def run_model(model, tokenizer, prompt, max_new_tokens=300, temperature=0.7, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
        **kwargs
    )
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# ────────────────────────────────────────────────
#   extract_json (current version)
# ────────────────────────────────────────────────
def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```json\s*|\s*```$', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'^(Here\'s|The function has been|formatted data:|JSON:).*?\{', '{', text, flags=re.IGNORECASE | re.DOTALL)

    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            print(f"JSON decode failed: {e} → raw: {json_str[:200]}...")
            return None

    print("No JSON object found in output")
    return None

# ────────────────────────────────────────────────
#   CHUNK FILTER
# ────────────────────────────────────────────────
def is_good_chunk(text):
    text = text.strip()
    if len(text) < 600: return False
    if len(text) > 4500: return False
    if text.count('\n') < 6: return False
    if len(text.split()) < 90: return False
    return True

good_indices = [i for i, c in enumerate(chunks) if is_good_chunk(c)]
print(f"Found {len(good_indices)} good chunks out of {len(chunks)} total\n")

# ────────────────────────────────────────────────
#   MAIN LOOP — with checkpoint every 10 samples
# ────────────────────────────────────────────────
dataset = []
target_samples = 100
max_attempts = 400
checkpoint_interval = 10
output_file = "my_first_ift_dataset.jsonl"

# Resume from existing file if it exists
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        dataset = [json.loads(line.strip()) for line in f if line.strip()]
    print(f"Loaded {len(dataset)} existing samples from {output_file}\n")

print("=== Starting / continuing IFT dataset creation ===\n")

attempt = 1
while len(dataset) < target_samples and attempt <= max_attempts:
    if not good_indices:
        print("No more good chunks left.")
        break

    idx = random.choice(good_indices)
    good_indices.remove(idx)
    chunk_text = chunks[idx]

    print(f"[{len(dataset)+1}/{target_samples}] Chunk #{idx} ({len(chunk_text)} chars) → ", end="")

    # 1. Facts
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FACTS},
        {"role": "user",   "content": f"Extract as many factual statements as possible from this text.\n\nTEXT:\n{chunk_text}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    raw_facts = run_model(model, tokenizer, prompt, max_new_tokens=400, temperature=0.1, do_sample=False)

    lines = [l.strip() for l in raw_facts.split('\n') if l.strip()]
    bullet_lines = [l for l in lines if l.startswith('- ')]

    if not bullet_lines:
        print("no bullets → skip")
        attempt += 1
        continue

    facts = "\n".join(bullet_lines)
    print("facts preview:", facts[:100].replace("\n", " ") + "...")

    # 2. IFT generation
    ift_prompt = IFT_GENERATION_PROMPT.format(facts=facts)
    raw_json_text = run_model(model, tokenizer, ift_prompt, max_new_tokens=350, temperature=0.3, do_sample=True)

    print("IFT raw:", raw_json_text[:300].replace("\n", " ").strip() + "...")

    sample_dict = extract_json(raw_json_text)

    if sample_dict is None:
        print("→ JSON parse failed → skip")
        attempt += 1
        continue

    # 3. Accept (self-eval disabled)
    dataset.append(sample_dict)
    print("→ ACCEPTED ✓")

    # Save immediately after every sample
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(sample_dict, ensure_ascii=False) + "\n")

    # Checkpoint message every 10
    if len(dataset) % checkpoint_interval == 0:
        print(f"\n*** CHECKPOINT *** Saved {len(dataset)} samples so far ***\n")

    attempt += 1

# Final summary
print("\n" + "="*70)
print(f"Finished — total: {len(dataset)} samples")
print(f"File: {output_file}")
print("="*70)

if dataset:
    print("\nLast sample:")
    print(json.dumps(dataset[-1], indent=2, ensure_ascii=False))

Found 153 good chunks out of 153 total

Loaded 47 existing samples from my_first_ift_dataset.jsonl

=== Starting / continuing IFT dataset creation ===

[48/100] Chunk #68 (1516 chars) → facts preview: - Developing Models via LLMOps - Figure 4.1 – Papers With Code – Question Answering leaderboard - Se...
IFT raw: """  def generate_instruction():     return {         "instruction": "What is the model used in Figure 4.1 - Papers With Code - Question Answering leaderboard?",         "input": "",         "response": "T5-11B"     }  # Generate instructions based on the provided code snippet instructions = [genera...
JSON decode failed: Extra data: line 7 column 1 (char 175) → raw: {
        "instruction": "What is the model used in Figure 4.1 - Papers With Code - Question Answering leaderboard?",
        "input": "",
        "response": "T5-11B"
    }

# Generate instructions b...
→ JSON parse failed → skip
[48/100] Chunk #95 (2537 chars) → facts preview: - The book covers advanced strategie